# Customer Support Ticket Classification: Analysis Notebook

This notebook walks through the full project: loading data, cleaning, EDA, training a
**TF-IDF + Logistic Regression** model, evaluating it, and predicting new tickets.

**Honest notes**
- The dataset is **synthetic** (written with AI assistance), not real company data.
- All results are computed by running the code below. Nothing is typed in by hand.
- AI assistance was used to help plan and write this project.

Run the cells from top to bottom (Kernel > Restart & Run All).

## 1. Setup

In [ ]:
import sys
sys.path.append("../src")   # lets the notebook import our project files

import joblib
import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (accuracy_score, classification_report, confusion_matrix,
                             f1_score, precision_score, recall_score)
from sklearn.model_selection import cross_val_score, train_test_split
from sklearn.pipeline import Pipeline

from preprocessing import DATA_PATH, clean_text, handle_missing_values, load_data, remove_duplicates
from predict import TEST_TICKETS, predict_ticket

sns.set_theme(style="whitegrid")
RANDOM_STATE = 42

## 2. Load and inspect the data

In [ ]:
raw_df = load_data(DATA_PATH)
raw_df.head()

In [ ]:
print("Shape (rows, columns):", raw_df.shape)
raw_df.info()

In [ ]:
print("Missing values per column:")
print(raw_df.isnull().sum())
print("\nDuplicate descriptions:", raw_df.duplicated(subset="ticket_description").sum())

**What we see:** some `priority` and `status` values are missing and some ticket descriptions
are repeated. These were added on purpose so we can practise cleaning.

## 3. Clean the data

1. Drop rows with no text or no category (we cannot learn from them).
2. Fill missing `priority` / `status` with "Unknown".
3. Remove duplicate descriptions (they would let the same ticket appear in train and test).
4. Lowercase, remove punctuation and extra spaces from the text.

In [ ]:
df = handle_missing_values(raw_df)
df = remove_duplicates(df)
df["clean_description"] = df["ticket_description"].apply(clean_text)
df = df.reset_index(drop=True)

print("Rows before:", len(raw_df), "| Rows after:", len(df))
df[["ticket_description", "clean_description"]].head()

## 4. Exploratory Data Analysis (EDA)

In [ ]:
print("Total tickets:", len(df))
for column in ["category", "priority", "status"]:
    print(f"\n{column.title()}-wise count")
    print(df[column].value_counts())
print("\nCategory distribution (%)")
print((df["category"].value_counts(normalize=True) * 100).round(1))

In [ ]:
plt.figure(figsize=(9, 5))
order = df["category"].value_counts().index
ax = sns.countplot(data=df, x="category", order=order, hue="category", palette="viridis", legend=False)
ax.bar_label(ax.containers[0])
plt.title("Ticket Category Distribution", weight="bold")
plt.xticks(rotation=15)
plt.show()

**Insight:** categories are fairly balanced (no category dominates), so accuracy is a fair metric
and we do not need special balancing.

In [ ]:
plt.figure(figsize=(7, 5))
ax = sns.countplot(data=df, x="priority", order=["High", "Medium", "Low", "Unknown"],
                   hue="priority", palette="rocket", legend=False)
ax.bar_label(ax.containers[0])
plt.title("Ticket Priority Distribution", weight="bold")
plt.show()

**Insight:** priorities are spread almost evenly across High, Medium and Low. Priority is synthetic
metadata here, so we should not treat this as real-world behaviour.

In [ ]:
plt.figure(figsize=(7, 5))
ax = sns.countplot(data=df, x="status", order=["Open", "In Progress", "Resolved", "Closed", "Unknown"],
                   hue="status", palette="mako", legend=False)
ax.bar_label(ax.containers[0])
plt.title("Ticket Status Distribution", weight="bold")
plt.show()

**Insight:** `Open` is the largest status, and Open + In Progress together are over half of all
tickets, which would suggest a backlog in a real support team.

In [ ]:
pd.crosstab(df["category"], df["priority"])

**Observation:** Login and Application Error tickets lean towards High priority, while Account Update and
Report tickets lean towards Low. This pattern was built into the synthetic data.

## 5. Train the model

- **TF-IDF** turns each ticket into numbers. Words that are distinctive get higher scores.
- **Logistic Regression** learns which words point to which category and outputs probabilities.
- A `Pipeline` fits the vectorizer **only on training data**, which prevents data leakage.

In [ ]:
X = df["clean_description"]
y = df["category"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=RANDOM_STATE, stratify=y
)
print("Training tickets:", len(X_train), "| Test tickets:", len(X_test))

model = Pipeline([
    ("tfidf", TfidfVectorizer(ngram_range=(1, 2))),
    ("classifier", LogisticRegression(max_iter=1000, random_state=RANDOM_STATE)),
])
model.fit(X_train, y_train)

## 6. Evaluate the model

In [ ]:
y_pred = model.predict(X_test)

print(f"Accuracy           : {accuracy_score(y_test, y_pred):.4f}")
print(f"Precision (macro)  : {precision_score(y_test, y_pred, average='macro'):.4f}")
print(f"Recall (macro)     : {recall_score(y_test, y_pred, average='macro'):.4f}")
print(f"F1 score (macro)   : {f1_score(y_test, y_pred, average='macro'):.4f}")
print()
print(classification_report(y_test, y_pred))

In [ ]:
class_names = sorted(y.unique())
matrix = confusion_matrix(y_test, y_pred, labels=class_names)

plt.figure(figsize=(8, 6))
sns.heatmap(matrix, annot=True, fmt="d", cmap="Blues",
            xticklabels=class_names, yticklabels=class_names)
plt.title("Confusion Matrix (Test Set)", weight="bold")
plt.xlabel("Predicted category")
plt.ylabel("Actual category")
plt.xticks(rotation=30, ha="right")
plt.tight_layout()
plt.show()

**How to read it:** each row is the real category and each column is what the model predicted.
Numbers on the diagonal are correct predictions. Numbers off the diagonal show which categories
get confused.

**Limitation:** the test set has only about 41 tickets, so one ticket changes accuracy by about 2.4%.
For a steadier estimate we also run cross-validation below.

In [ ]:
cv_scores = cross_val_score(model, X, y, cv=5, scoring="accuracy")
print("Cross-validation accuracy per fold:", [round(float(s), 3) for s in cv_scores])
print(f"Mean: {cv_scores.mean():.4f} (+/- {cv_scores.std():.4f})")

## 7. What did the model learn? (top words per category)

Logistic Regression gives each word a weight per category. The highest weights show which words
push a ticket towards that category.

In [ ]:
vectorizer = model.named_steps["tfidf"]
classifier = model.named_steps["classifier"]
feature_names = vectorizer.get_feature_names_out()

for index, category in enumerate(classifier.classes_):
    top_indices = classifier.coef_[index].argsort()[-6:][::-1]
    print(f"{category:<18}:", ", ".join(feature_names[top_indices]))

## 8. Predict new tickets

These tickets are new and not in `tickets.csv`.

In [ ]:
existing_texts = set(df["clean_description"])
rows = []
for text, expected in TEST_TICKETS:
    assert clean_text(text) not in existing_texts, "This ticket is in the training data!"
    predicted, confidence = predict_ticket(model, text)
    rows.append({
        "Input": text,
        "Expected": expected,
        "Predicted": predicted,
        "Confidence %": round(confidence, 1),
        "Result": "PASS" if predicted == expected else "FAIL",
    })

results = pd.DataFrame(rows)
print("Correct:", (results["Result"] == "PASS").sum(), "out of", len(results))
results

## 9. Save the model

In [ ]:
from pathlib import Path
model_path = Path("../model/trained_model.joblib")
model_path.parent.mkdir(exist_ok=True)
joblib.dump(model, model_path)
print("Saved to", model_path.resolve())

## 10. Limitations and future improvements

**Limitations**
- Small, synthetic dataset (about 200 tickets), so real-world accuracy would likely be lower.
- Tiny test set, so single-split metrics are noisy.
- Predicted probabilities are low and soft (with 5 classes, random guessing is 20%).
- Some tickets overlap categories, for example slow reports (Performance vs Report).
- Only English text and one label per ticket.

**Future improvements**
- Collect real, larger, labeled data and have multiple people check the labels.
- Try other models (Naive Bayes, Linear SVM) and tune settings with cross-validation.
- Add a "needs human review" rule for low-confidence predictions.
- Replace the rule-based auto-reply with an LLM-generated reply that a human reviews.